# Set 8 — Training CIFAR-10 classifiers for ForensicsApp

This notebook produces everything the ForensicsApp Set 8 tools load:

| Part | What | Files written to `set8_models/` |
|---|---|---|
| 8.1 | Small CNN trained from scratch | `cifar10_small_cnn.keras`, `cifar10_small_cnn.passport.json` |
| 8.1 | Transfer learning with InceptionV3 | `cifar10_inceptionv3.keras`, `cifar10_inceptionv3.passport.json` |
| 8.3 | Test-set feature vectors of both models | `cifar10_small_cnn.test_features.npz`, `cifar10_inceptionv3.test_features.npz` |

A `.keras` file holds the architecture **and** the trained weights. The **model passport** is a JSON file next to it
with the dataset and sample sizes, class names, exact preprocessing instructions, architecture, training history and
test results, so the app can show it and preprocess images without loading TensorFlow first.

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
At the end the folder is zipped and downloaded; unzip it into the app's `models/` folder.

## Setup

In [ ]:
import datetime
import json
import os
import platform

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras import layers

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| Python", platform.python_version())
print("GPU:", tf.config.list_physical_devices("GPU") or "none - switch the runtime to GPU")

OUT_DIR = "set8_models"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

## Data: CIFAR-10

50,000 training and 10,000 test images, 32×32 RGB, 10 classes. A fixed, seeded 5,000-image validation split is
taken from the training set; the official test set is only used for the final evaluation and the feature export.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_full, y_test = y_train_full.ravel(), y_test.ravel()

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

VAL_SIZE = 5000
perm = np.random.default_rng(SEED).permutation(len(x_train_full))
val_idx, train_idx = perm[:VAL_SIZE], perm[VAL_SIZE:]
x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

print("train", x_train.shape, "val", x_val.shape, "test", x_test.shape, x_train.dtype)
print("train images per class:", np.bincount(y_train))

fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for cl in range(10):
    for row, idx in enumerate(np.where(y_train == cl)[0][:3]):
        axes[row, cl].imshow(x_train[idx])
        axes[row, cl].axis("off")
    axes[0, cl].set_title(CLASS_NAMES[cl])
plt.show()

## Passport helpers

Every model is saved with the same passport layout. `preprocessing` is written as instructions the app follows
literally: convert to RGB, resize to `input_size` with bilinear interpolation, cast to float32, apply `formula`.

In [ ]:
def layer_table(model):
    return [{"name": l.name, "type": l.__class__.__name__, "parameters": int(l.count_params())}
            for l in model.layers]


def history_dict(*histories):
    # Concatenate the per-epoch metrics of one or more fit() calls; 'phase' says which call each epoch came from.
    merged = {"phase": []}
    for phase, history in enumerate(histories, start=1):
        for key, values in history.history.items():
            merged.setdefault(key, []).extend(float(v) for v in values)
        merged["phase"].extend([phase] * len(history.epoch))
    return merged


def evaluate(model, test_ds, labels):
    probs = model.predict(test_ds, verbose=0)
    preds = probs.argmax(axis=1)
    loss = float(keras.losses.SparseCategoricalCrossentropy()(labels, probs))
    per_class = {CLASS_NAMES[c]: float((preds[labels == c] == c).mean()) for c in range(10)}
    return {"test_accuracy": float((preds == labels).mean()), "test_loss": loss,
            "per_class_accuracy": per_class}, preds


def save_model_with_passport(model, name, description, preprocessing, training, evaluation, samples):
    model_file = f"{name}.keras"
    # Save an uncompiled copy that shares the trained layers: same weights, but no optimizer state
    # (Keras 3 ignores include_optimizer=False for .keras files; Adam's state would double the file size).
    keras.Model(model.inputs, model.outputs, name=model.name).save(os.path.join(OUT_DIR, model_file))
    passport = {
        "passport_version": 1,
        "name": name,
        "description": description,
        "model_file": model_file,
        "created": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "framework": {"tensorflow": tf.__version__, "keras": keras.__version__,
                      "python": platform.python_version()},
        "dataset": {
            "name": "CIFAR-10",
            "source": "keras.datasets.cifar10",
            "original_image_size": [32, 32, 3],
            "samples": samples,
            "split": f"seeded random {VAL_SIZE}-image validation split (seed {SEED}) from the 50,000 "
                     "training images; official 10,000-image test set",
        },
        "class_names": CLASS_NAMES,
        "preprocessing": preprocessing,
        "architecture": {
            "input_shape": list(model.input_shape[1:]),
            "parameters": int(model.count_params()),
            "trainable_parameters": int(sum(np.prod(w.shape) for w in model.trainable_weights)),
            "feature_layer": "features",
            "feature_dim": int(model.get_layer("features").output.shape[-1]),
            "layers": layer_table(model),
        },
        "training": training,
        "evaluation": evaluation,
    }
    passport_path = os.path.join(OUT_DIR, f"{name}.passport.json")
    with open(passport_path, "w") as f:
        json.dump(passport, f, indent=2)
    print("saved", model_file, "and", os.path.basename(passport_path))
    return passport


def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for key, ax in (("loss", ax1), ("accuracy", ax2)):
        ax.plot(history[key], label="train")
        ax.plot(history[f"val_{key}"], label="validation")
        ax.set_xlabel("epoch")
        ax.set_title(key)
        ax.legend()
    fig.suptitle(title)
    plt.show()

# 8.1a — Small CNN from scratch

A VGG-style network (the "deeper" model from the lab, extended to three blocks) with batch normalisation and dropout.
Input: 32×32 RGB scaled to [0, 1]. Random horizontal flips and small shifts are applied to the training batches only
(in the data pipeline, so the saved model itself has no augmentation layers).

In [ ]:
SMALL_PREPROCESSING = {
    "color_mode": "RGB",
    "input_size": [32, 32],
    "resize": "bilinear",
    "dtype": "float32",
    "formula": "x / 255.0",
    "scaled_range": [0.0, 1.0],
    "notes": "Resize the whole image to 32x32 (no cropping); add a batch dimension -> (1, 32, 32, 3).",
}
SMALL_BATCH, SMALL_EPOCHS, SMALL_LR = 64, 40, 0.01

augment = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomTranslation(0.1, 0.1)])


def small_dataset(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED)
    ds = ds.batch(SMALL_BATCH).map(lambda a, b: (tf.cast(a, tf.float32) / 255.0, b), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda a, b: (augment(a, training=True), b), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


def build_small_cnn():
    inputs = keras.Input((32, 32, 3), name="image")
    x = inputs
    for filters, drop in ((32, 0.2), (64, 0.3), (128, 0.4)):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding="same", activation="relu", kernel_initializer="he_uniform")(x)
            x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(drop)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu", kernel_initializer="he_uniform", name="features")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="cifar10_small_cnn")


keras.backend.clear_session()
small_model = build_small_cnn()
small_model.summary()

In [ ]:
small_model.compile(optimizer=keras.optimizers.SGD(SMALL_LR, momentum=0.9),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
small_fit = small_model.fit(
    small_dataset(x_train, y_train, training=True),
    validation_data=small_dataset(x_val, y_val),
    epochs=SMALL_EPOCHS,
    callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)],
)
small_history = history_dict(small_fit)
small_history.pop("learning_rate", None)  # added by ReduceLROnPlateau, not a metric
plot_history(small_history, "Small CNN")

In [ ]:
small_eval, small_preds = evaluate(small_model, small_dataset(x_test, y_test), y_test)
print(f"Small CNN test accuracy: {small_eval['test_accuracy']:.3f}")

small_passport = save_model_with_passport(
    small_model, "cifar10_small_cnn",
    "VGG-style CNN (3 blocks of 2 conv layers) trained from scratch on CIFAR-10.",
    SMALL_PREPROCESSING,
    {
        "optimizer": "SGD", "momentum": 0.9, "learning_rate": SMALL_LR, "batch_size": SMALL_BATCH,
        "loss": "sparse_categorical_crossentropy", "epochs_max": SMALL_EPOCHS,
        "epochs_run": len(small_history["loss"]),
        "callbacks": "EarlyStopping(patience=6, restore best weights), ReduceLROnPlateau(factor=0.5, patience=3)",
        "augmentation": "random horizontal flip, random translation up to 10% (training batches only)",
        "history": small_history,
    },
    small_eval,
    {"train": len(x_train), "validation": len(x_val), "test": len(x_test)},
)

# 8.1b — Transfer learning with InceptionV3

ImageNet-pretrained InceptionV3 without its top, followed by global average pooling and a small classifier head.
InceptionV3 needs at least 75×75 inputs, so CIFAR images are upsampled to `INC_SIZE` and scaled to [-1, 1]
(the same as `keras.applications.inception_v3.preprocess_input`).

* **Phase 1:** base frozen, train only the head (Adam, lr 1e-3).
* **Phase 2:** fine-tune the top two inception blocks (layers 249+) with a very low learning rate (1e-5).
  The base is always called with `training=False`, so its BatchNorm statistics stay fixed.

`TL_TRAIN_SAMPLES` limits the training images to keep Colab time reasonable; set it to `None` to use all 45,000.

In [ ]:
INC_SIZE = 139
TL_TRAIN_SAMPLES = 10000
TL_BATCH = 64
TL_EPOCHS_HEAD, TL_EPOCHS_FINE = 5, 5
TL_LR_HEAD, TL_LR_FINE = 1e-3, 1e-5

INCEPTION_PREPROCESSING = {
    "color_mode": "RGB",
    "input_size": [INC_SIZE, INC_SIZE],
    "resize": "bilinear",
    "dtype": "float32",
    "formula": "x / 127.5 - 1.0",
    "scaled_range": [-1.0, 1.0],
    "notes": f"Resize the whole image to {INC_SIZE}x{INC_SIZE} (no cropping); same scaling as "
             "keras.applications.inception_v3.preprocess_input; add a batch dimension.",
}

x_tl, y_tl = x_train[:TL_TRAIN_SAMPLES], y_train[:TL_TRAIN_SAMPLES]
print("transfer-learning train images:", len(x_tl), np.bincount(y_tl))


def inception_dataset(x, y, training=False):
    def prep(a, b):
        a = tf.image.resize(tf.cast(a, tf.float32), (INC_SIZE, INC_SIZE), method="bilinear")
        return a / 127.5 - 1.0, b

    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED)
    ds = ds.batch(TL_BATCH).map(prep, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda a, b: (tf.image.random_flip_left_right(a), b), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


keras.backend.clear_session()
base = keras.applications.InceptionV3(include_top=False, weights="imagenet", input_shape=(INC_SIZE, INC_SIZE, 3))
base.trainable = False

inputs = keras.Input((INC_SIZE, INC_SIZE, 3), name="image")
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu", name="features")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax", name="predictions")(x)
tl_model = keras.Model(inputs, outputs, name="cifar10_inceptionv3")
tl_model.summary()

In [ ]:
tl_train_ds = inception_dataset(x_tl, y_tl, training=True)
tl_val_ds = inception_dataset(x_val, y_val)

# Phase 1: train the head only
tl_model.compile(optimizer=keras.optimizers.Adam(TL_LR_HEAD),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
tl_fit_head = tl_model.fit(tl_train_ds, validation_data=tl_val_ds, epochs=TL_EPOCHS_HEAD)

# Phase 2: unfreeze the top two inception blocks and fine-tune with a very low learning rate
base.trainable = True
for layer in base.layers[:249]:
    layer.trainable = False
tl_model.compile(optimizer=keras.optimizers.Adam(TL_LR_FINE),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
tl_fit_fine = tl_model.fit(tl_train_ds, validation_data=tl_val_ds, epochs=TL_EPOCHS_FINE)

tl_history = history_dict(tl_fit_head, tl_fit_fine)
plot_history(tl_history, "InceptionV3 transfer learning (phase 1: head, phase 2: fine-tune)")

In [ ]:
tl_eval, tl_preds = evaluate(tl_model, inception_dataset(x_test, y_test), y_test)
print(f"InceptionV3 test accuracy: {tl_eval['test_accuracy']:.3f}")

tl_passport = save_model_with_passport(
    tl_model, "cifar10_inceptionv3",
    "ImageNet-pretrained InceptionV3 backbone + GAP + Dense(256) head, fine-tuned on CIFAR-10.",
    INCEPTION_PREPROCESSING,
    {
        "base_model": "InceptionV3 (ImageNet weights, include_top=False)",
        "optimizer": "Adam", "batch_size": TL_BATCH, "loss": "sparse_categorical_crossentropy",
        "augmentation": "random horizontal flip (training batches only)",
        "phases": [
            {"phase": 1, "trainable": "classifier head only", "learning_rate": TL_LR_HEAD,
             "epochs": len(tl_fit_head.epoch)},
            {"phase": 2, "trainable": "head + InceptionV3 layers 249 onwards (top two blocks)",
             "learning_rate": TL_LR_FINE, "epochs": len(tl_fit_fine.epoch)},
        ],
        "epochs_run": len(tl_history["loss"]),
        "history": tl_history,
    },
    tl_eval,
    {"train": len(x_tl), "validation": len(x_val), "test": len(x_test)},
)

### Confusion matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, preds, title in ((axes[0], small_preds, "Small CNN"), (axes[1], tl_preds, "InceptionV3")):
    ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=CLASS_NAMES, ax=ax,
                                            xticks_rotation=45, colorbar=False, cmap="Blues")
    ax.set_title(f"{title} (accuracy {np.mean(preds == y_test):.3f})")
plt.tight_layout()
plt.show()

# 8.3 — Export test-set feature vectors

For each model, the output of the `features` layer (the last hidden layer before the softmax) is computed for all
10,000 test images and saved with the true labels, the model's predictions and the class names.
The app loads these `.npz` files and projects them to 2D with t-SNE and UMAP, without needing the model.

`.npz` keys: `features` (N×D float32), `labels` (N), `predictions` (N), `class_names`, `model_name`, `layer_name`.

In [ ]:
def export_features(model, name, test_ds):
    extractor = keras.Model(model.inputs, [model.get_layer("features").output, model.output])
    features, probs = extractor.predict(test_ds, verbose=0)
    path = os.path.join(OUT_DIR, f"{name}.test_features.npz")
    np.savez_compressed(path, features=features.astype(np.float32), labels=y_test.astype(np.int64),
                        predictions=probs.argmax(axis=1).astype(np.int64), class_names=np.array(CLASS_NAMES),
                        model_name=np.array(name), layer_name=np.array("features"))
    print(f"saved {os.path.basename(path)}: features {features.shape}")
    return features


small_features = export_features(small_model, "cifar10_small_cnn", small_dataset(x_test, y_test))
tl_features = export_features(tl_model, "cifar10_inceptionv3", inception_dataset(x_test, y_test))

# Check: reload a saved model and classify with the passport (what the app does in 8.2)

The image is preprocessed with Pillow from the passport's instructions only, exactly as the app will do,
so a mismatch between training and app preprocessing would show up here.

In [ ]:
from PIL import Image


def preprocess_from_passport(image, passport):
    p = passport["preprocessing"]
    image = image.convert(p["color_mode"]).resize(tuple(p["input_size"]), Image.BILINEAR)
    x = np.asarray(image, dtype=np.float32)
    x = {"x / 255.0": x / 255.0, "x / 127.5 - 1.0": x / 127.5 - 1.0}[p["formula"]]
    return x[None]


for name in ("cifar10_small_cnn", "cifar10_inceptionv3"):
    with open(os.path.join(OUT_DIR, f"{name}.passport.json")) as f:
        passport = json.load(f)
    loaded = keras.models.load_model(os.path.join(OUT_DIR, passport["model_file"]))
    sample = np.random.default_rng(0).choice(len(x_test), 8, replace=False)
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
    for ax, idx in zip(axes, sample):
        probs = loaded.predict(preprocess_from_passport(Image.fromarray(x_test[idx]), passport), verbose=0)[0]
        ax.imshow(x_test[idx])
        ax.axis("off")
        ok = probs.argmax() == y_test[idx]
        ax.set_title(f"{passport['class_names'][probs.argmax()]} {probs.max():.0%}", color="green" if ok else "red")
    fig.suptitle(f"{name} (passport test accuracy {passport['evaluation']['test_accuracy']:.3f})")
    plt.show()

# Preview: t-SNE and UMAP of the exported features

A quick look at what the app's projection tool will show (3,000-image subsample for speed).

In [ ]:
%pip install -q umap-learn

In [ ]:
import umap
from sklearn.manifold import TSNE

subset = np.random.default_rng(SEED).choice(len(y_test), 3000, replace=False)
fig, axes = plt.subplots(2, 2, figsize=(14, 13))
for row, (name, feats) in enumerate((("Small CNN", small_features), ("InceptionV3", tl_features))):
    projections = {
        "t-SNE": TSNE(n_components=2, perplexity=40, init="pca", random_state=SEED).fit_transform(feats[subset]),
        "UMAP": umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(feats[subset]),
    }
    for col, (method, xy) in enumerate(projections.items()):
        ax = axes[row, col]
        for cl in range(10):
            pts = xy[y_test[subset] == cl]
            ax.scatter(pts[:, 0], pts[:, 1], s=4, label=CLASS_NAMES[cl])
        ax.set_title(f"{name} — {method}")
        ax.set_xticks([])
        ax.set_yticks([])
axes[0, 1].legend(markerscale=4, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Download the results

Unzip `set8_models.zip` into the ForensicsApp `models/` folder (it is git-ignored, so the weights are not committed).
Optionally also copy the folder to Google Drive.

In [ ]:
import shutil

for f in sorted(os.listdir(OUT_DIR)):
    print(f"{os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:8.1f} MB  {f}")

zip_path = shutil.make_archive("set8_models", "zip", root_dir=".", base_dir=OUT_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; zip saved at", os.path.abspath(zip_path))

In [ ]:
# Optional: keep a copy in Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(OUT_DIR, "/content/drive/MyDrive/set8_models", dirs_exist_ok=True)